<a href="https://colab.research.google.com/github/Abody-elneel/Abody-elneel/blob/main/notebook04b7cc6991.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import base64
from download import download
url_train =\
b"aHR0cHM6Ly9hc2NlbmQtcHJvZmVzc2lvbmFsLWNvbnN0cnVjdGlvbi1kYXRhc2V0Lm9icy5teWh1YXdlaWNsb3VkLmNvbS9kZWVwLWxlYXJuaW5nL2Zsb3dlcl9waG90b3NfdHJhaW4uemlw"

url_test =\
b"aHR0cHM6Ly9hc2NlbmQtcHJvZmVzc2lvbmFsLWNvbnN0cnVjdGlvbi1kYXRhc2V0Lm9icy5teWh1YXdlaWNsb3VkLmNvbS9kZWVwLWxlYXJuaW5nL2Zsb3dlcl9waG90b3NfdGVzdC56aXA="

path_train = download(base64.b64decode(url_train).decode(), "./", kind="zip", replace=True, verbose=False)
path_test = download(base64.b64decode(url_test).decode(), "./", kind="zip", replace=True, verbose=False)

file_sizes: 100%|████████████████████████████| 227M/227M [00:32<00:00, 6.88MB/s]
file_sizes: 100%|██████████████████████████| 3.01M/3.01M [00:01<00:00, 1.96MB/s]


In [2]:
!pip install download

In [4]:
import torch
import torchvision
import torch.nn.functional as F
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from easydict import EasyDict as edict
import random

In [11]:
cfg = edict({
    'data_path': './flower_photos_train', # Path of the training dataset
    'test_path':'./flower_photos_test',     #Path of the test dataset
    'data_size': 3616,
    'HEIGHT': 224, # Image height
    'WIDTH': 224, # Image width
    '_R_MEAN': 123.68, # Mean of CIFAR10
    '_G_MEAN': 116.78,
    '_B_MEAN': 103.94,
    '_R_STD': 1, # User-defined standard deviation
    '_G_STD': 1,
    '_B_STD':1,
    '_RESIZE_SIDE_MIN': 256, # Minimum resize value of image enhancement
    '_RESIZE_SIDE_MAX': 512,

    'batch_size': 1, # Batch size
    'num_class': 5, # Classification
    'epoch_size': 5, # Number of training times
    'num_workers':1,
    'device':"cpu", # cpu, xpu, etc.

    'prefix': ' renet50.pth',  # Model name

})

In [12]:
# Use transforms.Compose to combine multiple processing modes.
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),
    transforms.Normalize(mean=[cfg._R_MEAN, cfg._G_MEAN, cfg._B_MEAN], std=[cfg._R_STD, cfg._G_STD,
cfg._B_STD]),
    transforms.Resize(cfg._RESIZE_SIDE_MIN),
    transforms.CenterCrop((cfg.HEIGHT, cfg.WIDTH)),

])

transform_test = transforms.Compose([
      transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [13]:
train = torchvision.datasets.ImageFolder(cfg. data_path, transform=transform_train)
trainloader = torch.utils.data.DataLoader(train, batch_size=cfg.batch_size, shuffle=True,
num_workers=cfg.num_workers)

In [14]:
import torch.nn as nn

# Define the basic block, which is mainly used for ResNet-18 and ResNet-34.
class BasicBlock(nn.Module):
    expansion = 1  # Multiple for expansion of the number of output channels

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()

        # Residual function
        self.residual_function = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),  # The first convolutional layer
            nn.BatchNorm2d(out_channels),  # The first batch normalization layer
            nn.ReLU(inplace=True),  # Activation function
            nn.Conv2d(out_channels, out_channels * BasicBlock.expansion, kernel_size=3, padding=1, bias=False),  # The second convolutional layer
            nn.BatchNorm2d(out_channels * BasicBlock.expansion)  # The second batch normalization layer
        )

        # Shortcut, which is used to adjust the dimensions and stride
        self.shortcut = nn.Sequential()

        # If the dimensions do not match, use 1x1 convolution to adjust the dimensions of the shortcut.
        if stride != 1 or in_channels != BasicBlock.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * BasicBlock.expansion)
            )

    def forward(self, x):
        # Residual connection and activation function
        return nn.ReLU(inplace=True)(self.residual_function(x) + self.shortcut(x))


# Define the bottleneck block, which is mainly used for ResNet-50, ResNet-101, and ResNet-152.
class BottleNeck(nn.Module):
    expansion = 4  # Multiple for expansion of the number of output channels

    def __init__(self, in_channels, out_channels, stride=1):
        super(BottleNeck, self).__init__()

        # Residual function
        self.residual_function = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),  # The first 1x1 convolutional layer for reducing dimensions
            nn.BatchNorm2d(out_channels),  # The first batch normalization layer
            nn.ReLU(inplace=True),  # Activation function
            nn.Conv2d(out_channels, out_channels, stride=stride, kernel_size=3, padding=1, bias=False),  # 3x3 convolutional layer
            nn.BatchNorm2d(out_channels),  # The second batch normalization layer
            nn.ReLU(inplace=True),  # Activation function
            nn.Conv2d(out_channels, out_channels * BottleNeck.expansion, kernel_size=1, bias=False),  # The second 1x1 convolutional layer for adding dimensions
            nn.BatchNorm2d(out_channels * BottleNeck.expansion)  # The third batch normalization layer
        )

        # Shortcut, which is used to adjust the dimensions and stride
        self.shortcut = nn.Sequential()

        # If the dimensions do not match, use 1x1 convolution to adjust the dimensions of the shortcut.
        if stride != 1 or in_channels != out_channels * BottleNeck.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * BottleNeck.expansion, stride=stride, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels * BottleNeck.expansion)
            )

    def forward(self, x):
        # Residual connection and activation function
        return nn.ReLU(inplace=True)(self.residual_function(x) + self.shortcut(x))


# Define the ResNet network.
class ResNet(nn.Module):

    def __init__(self, block, num_block, num_classes=100):
        super(ResNet, self).__init__()

        self.in_channels = 64  # Initial number of channels

        # The first convolutional layer
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        # The number of layers varies depending on the ResNet version.
        self.conv2_x = self._make_layer(block, 64, num_block[0], 1)
        self.conv3_x = self._make_layer(block, 128, num_block[1], 2)
        self.conv4_x = self._make_layer(block, 256, num_block[2], 2)
        self.conv5_x = self._make_layer(block, 512, num_block[3], 2)

        # Global average pooling layer
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully-connected layer. Set the number of output classes as required.
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        # Generate a sequence of layers.
        strides = [stride] + [1] * (num_blocks - 1)  # Set the stride of each block to 1 except the first block.
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))  # Add a block.
            self.in_channels = out_channels * block.expansion  # Update the number of channels.

        return nn.Sequential(*layers)  # Return the layer sequence.

    def forward(self, x):
        # Forward propagation
        output = self.conv1(x)
        output = self.conv2_x(output)
        output = self.conv3_x(output)
        output = self.conv4_x(output)
        output = self.conv5_x(output)
        output = self.avg_pool(output)  # Global average pooling
        output = output.view(output.size(0), -1)  # Flatten into a one-dimensional vector.
        output = self.fc(output)  # Fully-connected layer

        return output


In [15]:
# Define ResNets of different versions.
def resnet18():
    """Return the ResNet-18 model"""
    return ResNet(BasicBlock, [2, 2, 2, 2])

def resnet34():
    """Return the ResNet-34 model"""
    return ResNet(BasicBlock, [3, 4, 6, 3])

def resnet50():
    """Return the ResNet-50 model"""
    return ResNet(BottleNeck, [3, 4, 6, 3], num_classes=cfg.num_class)  # Change the number of classes.

def resnet101():
    """Return the ResNet-101 model"""
    return ResNet(BottleNeck, [3, 4, 23, 3])

def resnet152():
    """Return the ResNet-152 model"""
    return ResNet(BottleNeck, [3, 8, 36, 3])


In [16]:
import os

net = resnet50().to(cfg.device)
if os.path.isfile(cfg.prefix):
    net.load_state_dict(torch.load(cfg.prefix))


criterion = nn.CrossEntropyLoss()

In [17]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=2)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
for epoch in range(cfg.epoch_size):
    losses = []
    running_loss = 0.0

    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(cfg.device), labels.to(cfg.device)
        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        losses.append(loss.item())

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 200 == 0 and i > 0:
            print(f'Loss [epoch {epoch+1}, minibatch {i}]: {running_loss / 200:.4f}')
            running_loss = 0.0

    avg_loss = sum(losses) / len(losses)
    scheduler.step(avg_loss)

print('Training Done')


KeyboardInterrupt: 

In [ ]:
torch.save(net.state_dict(),cfg. prefix)

In [ ]:
test = torchvision.datasets.ImageFolder(cfg. test_path, transform=transform_test)
testloader = torch.utils.data.DataLoader(test, batch_size=cfg.batch_size, shuffle=True,
num_workers=cfg.num_workers)

correct = 0
total = 0

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images, labels = images.to(cfg.device), labels.to(cfg.device)
        outputs = net(images)

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        print('Accuracy: ', 100*(correct/total), '%')

Accuracy:  0.0 %
Accuracy:  0.0 %
Accuracy:  33.33333333333333 %
Accuracy:  25.0 %
Accuracy:  20.0 %
Accuracy:  16.666666666666664 %
Accuracy:  14.285714285714285 %
Accuracy:  12.5 %
Accuracy:  11.11111111111111 %
Accuracy:  10.0 %
Accuracy:  9.090909090909092 %
Accuracy:  8.333333333333332 %
Accuracy:  7.6923076923076925 %
Accuracy:  7.142857142857142 %
Accuracy:  6.666666666666667 %
Accuracy:  6.25 %
Accuracy:  5.88235294117647 %
Accuracy:  5.555555555555555 %
Accuracy:  5.263157894736842 %
Accuracy:  10.0 %
Accuracy:  9.523809523809524 %
Accuracy:  9.090909090909092 %
Accuracy:  8.695652173913043 %
Accuracy:  8.333333333333332 %
Accuracy:  12.0 %
Accuracy:  11.538461538461538 %
Accuracy:  11.11111111111111 %
Accuracy:  10.714285714285714 %
Accuracy:  10.344827586206897 %
Accuracy:  10.0 %
Accuracy:  12.903225806451612 %
Accuracy:  15.625 %
Accuracy:  15.151515151515152 %
Accuracy:  17.647058823529413 %
Accuracy:  17.142857142857142 %
Accuracy:  19.444444444444446 %
Accuracy:  18.918

In [5]:
!pip install -q albumentations==1.3.1
import os, cv2, random, albumentations as A
from PIL import Image
from glob import glob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 8.3 MB/s eta 0:00:00


In [6]:
transform = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=25, p=0.9),
    A.OneOf([
        A.Blur(blur_limit=3, p=1),
        A.GaussNoise(var_limit=(5, 10), p=1)
    ], p=0.3),
    A.Resize(224, 224)   # keeps uniform size
])

In [7]:
ROOT_DIR = "flower_photos_train"
EXTRA_PER_CLASS = 3_000

transform = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=25, p=0.9),
    A.OneOf([A.Blur(blur_limit=3, p=1), A.GaussNoise(var_limit=(5, 10), p=1)], p=0.3),
    A.Resize(224, 224)
])

for cls in os.listdir(ROOT_DIR):
    cls_path = os.path.join(ROOT_DIR, cls)
    originals = glob(os.path.join(cls_path, "*"))

    if not originals:
        print(f"⚠️  Skipping empty folder: {cls_path}")
        continue

    for idx in range(EXTRA_PER_CLASS):
        src = random.choice(originals)
        img = cv2.imread(src)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        aug_img = transform(image=img)["image"]
        save_path = os.path.join(cls_path, f"aug_{idx:05d}.jpg")
        Image.fromarray(aug_img).save(save_path)

print("✅ Augmentation finished.")

⚠️  Skipping empty folder: flower_photos_train/LICENSE.txt
✅ Augmentation finished.


In [10]:
import os

root = "flower_photos_train"
for cls in sorted(os.listdir(root)):
    cls_path = os.path.join(root, cls)
    if not os.path.isdir(cls_path):
        continue
    n = len(os.listdir(cls_path))
    print(f"{cls:>12}: {n} files")

       daisy: 3623 files
   dandelion: 3886 files
       roses: 3631 files
  sunflowers: 3689 files
      tulips: 3789 files
